In [1]:
import rqdatac
rqdatac.init('15805988503', 'Ss888888')
import numpy as np
import pandas as pd
import datetime as dt
import os
import matplotlib.pyplot as plt
from tqdm import tqdm_notebook, tnrange
import json
import datetime

# 基础数据

In [2]:
# 全A股数据
all_gotPrice = pd.read_csv('all_gotPrice.csv')
all_gotPrice['date'] = pd.to_datetime(all_gotPrice['date'])
# 中证500数据
zz500 = pd.read_csv('zz500.csv')
zz500.reset_index(inplace=True)
zz500['date'] = pd.to_datetime(zz500['date'])
# 交易日
tradeDates = all_gotPrice['date'].unique()
tradeDates = pd.to_datetime(tradeDates)
tradeDates.sort_values()
display(all_gotPrice.columns)
# 代码调整
adjust_codes = []
for code in tqdm_notebook(all_gotPrice['order_book_id']):
    adjust_code = code.split('.')[0]
    adjust_codes.append(adjust_code)
all_gotPrice['stk_code'] = adjust_codes

is_st = pd.read_csv('is_st.csv')
is_suspended = pd.read_csv('is_suspended.csv')
is_st.rename(columns={'index': 'date'}, inplace=True)
is_suspended.rename(columns={'index': 'date'}, inplace=True)
is_st['date'] = pd.to_datetime(is_st['date'])
is_suspended['date'] = pd.to_datetime(is_suspended['date'])

C:\Users\stansfield\AppData\Local\Temp\ipykernel_29368\3312505086.py:2: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  all_gotPrice = pd.read_csv('all_gotPrice.csv')


Index(['Unnamed: 0', 'order_book_id', 'industry_code', 'market_tplus',
       'symbol', 'special_type', 'exchange', 'status', 'type',
       'de_listed_date', 'listed_date', 'sector_code_name', 'abbrev_symbol',
       'sector_code', 'round_lot', 'trading_hours', 'board_type',
       'industry_name', 'issue_price', 'trading_code', 'office_address',
       'province', 'purchasedate', 'date', 'total_turnover', 'low', 'volumn',
       'limit_up', 'close', 'limit_down', 'prev_close', 'num_trades', 'open',
       'high', 'stk_code', 'over3months', 'index'],
      dtype='object')

C:\Users\stansfield\AppData\Local\Temp\ipykernel_29368\3312505086.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for code in tqdm_notebook(all_gotPrice['order_book_id']):


  0%|          | 0/9168476 [00:00<?, ?it/s]

# 回测框架
## 净值
判断是否调仓日：是，调仓后收盘价加总；否，当日收盘价加总。

In [22]:
def find_nearest_trading_date(trading_dates, target_date):
    # 将目标日期转换为Timestamp对象
    target_date = pd.Timestamp(target_date)

    # 遍历交易日期列表，找到最接近的日期
    nearest_date = min(trading_dates, key=lambda date: abs(date - target_date))

    return nearest_date, trading_dates.get_loc(nearest_date)

def find_nearest_latter_trading_date(trading_dates, target_date):
    # 将目标日期转换为Timestamp对象
    target_date = pd.Timestamp(target_date)

    # 遍历交易日期列表，找到最接近的日期
    nearest_date = min(trading_dates, key=lambda date: abs(date - target_date))
    if nearest_date < target_date:
        nearest_date = trading_dates[trading_dates.get_loc(nearest_date) + 1]

    return nearest_date, trading_dates.get_loc(nearest_date)

## 米筐功能函数

In [ ]:
class context(object):
    def __init__(self):
        self.now = None
        self.portfolio = None

class now(object):
    def __init__(self):
        self.date = None

class portfolio(object):
    def __init__(self):
        self.positions = None

## 米筐策略函数

In [ ]:
def handle_bar(context, bar_dict):
    # 当前日期
    cur_date = context.now.date()
    
    # 指定日期调仓
    if str(cur_date) in data.index:
        # 指定日期要买入的股票池
        stock_list = data.loc[str(cur_date)].tolist()
        # 过滤当日停牌个股
        stock_list = [code for code in stock_list if not is_suspended(code)]
        # 计算个股总得分
        score = get_total_score(stock_list, cur_date, days)
        # 买入得分小于max_score的个股
        buy_list = score[score < max_score].index.tolist()
        
        # 调仓
        # 卖出
        for code in context.portfolio.positions:
            order_target_percent(code, 0)
        # 买入
        for code in buy_list:
            order_target_percent(code, float(1/len(buy_list)))
    
    # 持仓个股
    holding_stocks = list(context.portfolio.positions)
    if holding_stocks:
        # 持仓个股总得分
        score = get_total_score(holding_stocks, cur_date, days)
        # 卖出得分大于等于max_score的个股
        sell_list = score[score >= max_score].index.tolist()
        for code in sell_list:
            order_target_percent(code, 0)
        
    # 止盈/止损
    stop_loss(context)

def get_total_score(stock_list, date, days):
    # 价格数据
    close = rqdatac.get_price(stock_list, start_date=rqdatac.get_previous_trading_date(date, days+200), end_date=date, frequency='1d', fields=['close'])
    close = pd.pivot_table(close, values='close', index='date', columns='order_book_id')
    high = rqdatac.get_price(stock_list, start_date=rqdatac.get_previous_trading_date(date, days+200), end_date=date, frequency='1d', fields=['high'])
    high = pd.pivot_table(high, values='high', index='date', columns='order_book_id')
    low = rqdatac.get_price(stock_list, start_date=rqdatac.get_previous_trading_date(date, days+200), end_date=date, frequency='1d', fields=['low'])
    low = pd.pivot_table(low, values='low', index='date', columns='order_book_id')
    
    # 计算均线
    MA5 = close.rolling(window=5).mean()
    MA20 = close.rolling(window=20).mean()
    MA30 = close.rolling(window=30).mean()
    # 计算MACD
    dif = close.apply(lambda x: cal_MACD(x)[0])
    dea = close.apply(lambda x: cal_MACD(x)[1])
    
    #1. 个股股价在前1个月内是否出现过向下突破20日均线
    con1 = ((close[-days:] < MA20[-days:]) & (close.shift(1)[-days:] > MA20.shift(1)[-days:])).sum()
    con1[con1 != 0] = 1
    #2. 个股股价在前1个月内的5日简单均线是否出现过向下突破30日简单均线
    con2 = ((MA5[-days:] < MA30[-days:]) & (MA5.shift(1)[-days:] > MA30.shift(1)[-days:])).sum()
    con2[con2 != 0] = 1
    #3. 个股前1个月是否出现过 DIF向下突破DEA（MACD由正转负）
    con3 = ((dif[-days:] < dea[-days:]) & (dif.shift(1)[-days:] > dea.shift(1)[-days:])).sum()
    con3[con3 != 0] = 1
    #4. 20日乖离率是否超过 8%
    bias20 = close.iloc[-1, :] / MA20.iloc[-1, :] - 1
    con4 = (bias20 > 0.08).astype(int)
    #5. KDJ中的K和D至少有一个大于80
    con5 = pd.Series(0, index=con1.index)
    for code in con5.index:
        k, d, j = cal_KDJ(close[code], high[code], low[code])
        if k[-1] > 80 or d[-1] > 80:
            con5[code] = 1
    # 计算总得分
    score = con1 + con2 + con3 + con4 + con5
    
    return score


# MACD
def cal_MACD(CLOSE, SHORT=12, LONG=26, M=9):
    DIF = cal_EMA(CLOSE,SHORT) - cal_EMA(CLOSE,LONG)
    DEA = cal_EMA(DIF,M)
    MACD = (DIF-DEA)*2
    
    return DIF, DEA, MACD


# KDJ
def cal_KDJ(CLOSE, HIGH, LOW, N=9, M1=3, M2=3):
    RSV = (CLOSE - cal_LLV(LOW, N)) / (cal_HHV(HIGH, N) - cal_LLV(LOW, N)) * 100
    K = cal_EMA(RSV, (M1*2-1))
    D = cal_EMA(K, (M2*2-1))
    J = K*3 - D*2
    
    return K, D, J


# EMA
def cal_EMA(S,N):
    return pd.Series(S).ewm(span=N, adjust=False).mean().values

# N日最大值
def cal_HHV(S,N):     
    return pd.Series(S).rolling(N).max().values

# N日最小值
def cal_LLV(S,N):
    return pd.Series(S).rolling(N).min().values
        

## 米筐回测程序

In [ ]:
#读取多因子选出的股票
XGBoost = pd.read_csv('./portfolio1/portfolio_df.csv', index_col = 0)

Factor_dfs = [XGBoost]
Factor_df_names = ['XGBoost']
#分年回测
back_test_details = pd.DataFrame()

data = XGBoost
df_name = 'XGBoost'
start_date = '2019-01-31'
end_date = '2024-05-16'
# 初始资金
init_cash = 1000000
# 基准
benchmark = '000905.XSHG'
# 手续费(%)
commission = 0.3
min_commission = 5
tax = 0.1

# 改变日期格式
data.index = pd.to_datetime(data.index.tolist())
# 判断买入条件的天数
days = 22
# 得分上限
max_score = 3
# 止盈/止损阀值
win_threshold = 0.20
loss_threshold = 0.15
# 回测
config = {"base": {"accounts": {"STOCK": init_cash}, "start_date": start_date, "end_date": end_date, "frequency": "1d"},
        "mod": {"sys_analyser": {"plot": True, "benchmark": benchmark},
                "sys_transaction_cost": {"stock_commission_multiplier ": commission, "tax_multiplier": tax, "cn_stock_min_commission": min_commission}},
        "extra": {"log_level": "error"}}
result = run_func(init=init, before_trading=before_trading, handle_bar=handle_bar, after_trading=after_trading, config=config)
#获取回测结果详细分析
result_analysis = result['sys_analyser']
#获取交易历史记录
trade_records = result_analysis['trades']
trade_records.to_csv(f"超预期_trade_records.csv")
#获取portfolio
portfolio = result_analysis['portfolio']
benchmark = result_analysis['benchmark_portfolio']
concat_port_ben = pd.concat([portfolio,benchmark],axis = 1)
concat_port_ben.to_csv(f'超预期_portfolio_records.csv')
#获取summray并合并进back_test_details
summary = pd.DataFrame(result_analysis['summary']).iloc[0].to_frame()
back_test_details = pd.concat([back_test_details,summary],axis = 1)
back_test_details.to_csv(f"超预期分年回测summary.csv")

# Backtrader

In [ ]:
with open ('portfolio_dict1_withForecast.json', 'r') as f:
    portfolio_dict = json.load(f)

trade_date = []
sec_code = []
weight = []

for key in portfolio_dict.keys():
    trade_date += [key] * len(portfolio_dict[key])
    weight += [1/len(portfolio_dict[key])] * len(portfolio_dict[key])
    for stk_code in portfolio_dict[key]:
        sec_code.append(rqdatac.id_convert(stk_code))

portfolio_bt = pd.DataFrame({'trade_date': trade_date, 'sec_code': sec_code, 'weight': weight})
portfolio_bt.to_csv('portfolio_bt.csv', index=False)


In [3]:
import backtrader as bt # 导入 Backtrader
import backtrader.indicators as btind # 导入策略分析模块
import backtrader.feeds as btfeeds # 导入数据模块

daily_price_bt = pd.DataFrame({'datetime': all_gotPrice['date'], 'order_book_id': all_gotPrice['order_book_id'],'open': all_gotPrice['open'], 'high': all_gotPrice['high'], 'low': all_gotPrice['low'], 'close': all_gotPrice['close'], 'volume': all_gotPrice['volumn']})
daily_price_bt['openinterest'] = [0]*daily_price_bt.shape[0]
daily_price_bt = daily_price_bt[daily_price_bt['datetime'] >= '2019-01-31']
daily_price_bt.set_index('datetime', inplace=True)

trade_info = pd.read_csv('portfolio_bt.csv')

zz500_bt = pd.DataFrame({'datetime': zz500['date'], 'open': zz500['open'], 'high': zz500['high'], 'low': zz500['low'], 'close': zz500['close'], 'volume': zz500['volume']})
zz500_bt['openinterest'] = [0]*zz500_bt.shape[0]
zz500_bt = zz500_bt[zz500_bt['datetime'] >= '2019-01-31']
zz500_bt.set_index('datetime', inplace=True)

start_date = pd.to_datetime('2019-01-31')
end_date = pd.to_datetime('2024-05-16')

# 创建策略
class TestStrategy(bt.Strategy):
    '''选股策略'''
    def __init__(self, trade_info):
        self.buy_stock = trade_info # 保留调仓列表
        # 读取调仓日期，即每月的最后一个交易日，回测时，会在这一天下单，然后在下一个交易日，以开盘价买入
        self.trade_dates = pd.to_datetime(self.buy_stock['trade_date'].unique()).tolist()
        self.order_list = [] # 记录以往订单，方便调仓日对未完成订单做处理
        self.buy_stocks_pre = [] # 记录上一期持仓
    def next(self):
        dt = self.datas[0].datetime.date(0) # 获取当前的回测时间点
        print("当前时间点：", dt)
        # 如果是调仓日，则进行调仓操作
        if dt in self.trade_dates:
            print("--------------{} 为调仓日----------".format(dt))
            # 在调仓之前，取消之前所下的没成交也未到期的订单
            if len(self.order_list) > 0:
                for od in self.order_list:
                    self.cancel(od) # 如果订单未完成，则撤销订单
                self.order_list = [] #重置订单列表
            # 提取当前调仓日的持仓列表
            buy_stocks_data = self.buy_stock.query(f"trade_date=='{dt}'")
            long_list = buy_stocks_data['sec_code'].tolist()
            print('long_list', long_list) # 打印持仓列表
            # 对现有持仓中，调仓后不再继续持有的股票进行卖出平仓
            sell_stock = [i for i in self.buy_stocks_pre if i not in long_list]
            print('sell_stock', sell_stock) # 打印平仓列表
            if len(sell_stock) > 0:
                print("-----------对不再持有的股票进行平仓--------------")
                for stock in sell_stock:
                    data = self.getdatabyname(stock)
                    if self.getposition(data).size > 0 :
                        od = self.close(data=data)
                        self.order_list.append(od) # 记录卖出订单
            # 买入此次调仓的股票：多退少补原则
            print("-----------买入此次调仓期的股票--------------")
            for stock in long_list:
                w = buy_stocks_data.query(f"sec_code=='{stock}'")['weight'].iloc[0] # 提取持仓权重
                data = self.getdatabyname(stock)
                order = self.order_target_percent(data=data, target=w*0.95) # 为减少可用资金不足的情况，留 5% 的现金做备用
                self.order_list.append(order)
       
            self.buy_stocks_pre = long_list # 保存此次调仓的股票列表
        
# 实例化 cerebro
cerebro = bt.Cerebro()
# 添加策略
cerebro.addstrategy(TestStrategy, trade_info=trade_info)


# 加上市场数据
for stock in tqdm_notebook(daily_price_bt['order_book_id'].unique()):
    # 日期对齐
    data = pd.DataFrame(index=daily_price_bt.index.unique()) # 获取回测区间内所有交易日
    df = daily_price_bt.query(f"order_book_id=='{stock}'")[['open','high','low','close','volume','openinterest']]
    data_ = pd.merge(data, df, left_index=True, right_index=True, how='left')
    # 缺失值处理：日期对齐时会使得有些交易日的数据为空，所以需要对缺失数据进行填充
    data_.loc[:,['volume','openinterest']] = data_.loc[:,['volume','openinterest']].fillna(0)
    data_.loc[:,['open','high','low','close']] = data_.loc[:,['open','high','low','close']].fillna(method='pad')
    data_.loc[:,['open','high','low','close']] = data_.loc[:,['open','high','low','close']].fillna(0)
    # 导入数据
    datafeed = bt.feeds.PandasData(dataname=data_, fromdate=start_date, todate=end_date)
    cerebro.adddata(datafeed, name=stock) # 通过 name 实现数据集与股票的一一对应

# 设置中证500为benchmark
data_ = zz500_bt
# 缺失值处理：日期对齐时会使得有些交易日的数据为空，所以需要对缺失数据进行填充
data_.loc[:,['volume','openinterest']] = data_.loc[:,['volume','openinterest']].fillna(0)
data_.loc[:,['open','high','low','close']] = data_.loc[:,['open','high','low','close']].fillna(method='pad')
data_.loc[:,['open','high','low','close']] = data_.loc[:,['open','high','low','close']].fillna(0)
# 导入数据
benchmark = bt.feeds.PandasData(dataname=data_, fromdate=start_date, todate=end_date)
cerebro.adddata(benchmark, name='CSI 500')
cerebro.addobserver(bt.observers.Benchmark, data=benchmark)

# 通过经纪商设置初始资金
cerebro.broker.setcash(1000000.0)
# 设置交易佣金
cerebro.broker.setcommission(0.003)
# 设置滑点
cerebro.broker.set_slippage_perc(perc=0.0001)

# 添加分析器
cerebro.addanalyzer(bt.analyzers.TimeReturn, _name='pnl') # 返回收益率时序数据
cerebro.addanalyzer(bt.analyzers.AnnualReturn, _name='_AnnualReturn') # 年化收益率
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='_SharpeRatio') # 夏普比率
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='_DrawDown') # 回撤
cerebro.addanalyzer(bt.analyzers.TimeReturn, _name='_TimeReturn')

C:\Users\stansfield\AppData\Local\Temp\ipykernel_29368\1775180151.py:71: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for stock in tqdm_notebook(daily_price_bt['order_book_id'].unique()):


  0%|          | 0/5259 [00:00<?, ?it/s]

In [4]:
result = cerebro.run()
print('done')
# 可视化回测结果
#cerebro.plot()

# 从返回的 result 中提取回测结果
strat = result[0]
# 返回日度收益率序列
daily_return = pd.Series(strat.analyzers.pnl.get_analysis())
# 打印评价指标
print("--------------- AnnualReturn -----------------")
print(strat.analyzers._AnnualReturn.get_analysis())
print("--------------- SharpeRatio -----------------")
print(strat.analyzers._SharpeRatio.get_analysis())
print("--------------- DrawDown -----------------")
print(strat.analyzers._DrawDown.get_analysis())

当前时间点： 2019-01-31
--------------2019-01-31 为调仓日----------
long_list ['600519.XSHG', '300347.XSHE', '300073.XSHE', '600699.XSHG', '002299.XSHE', '002916.XSHE', '000401.XSHE', '002475.XSHE', '300699.XSHE', '300394.XSHE', '300457.XSHE', '600567.XSHG', '600845.XSHG', '300673.XSHE', '300628.XSHE', '000539.XSHE', '000672.XSHE', '002353.XSHE', '300575.XSHE', '300735.XSHE', '601012.XSHG', '601668.XSHG', '603338.XSHG', '603588.XSHG', '002798.XSHE', '002831.XSHE', '300284.XSHE', '600019.XSHG', '601058.XSHG', '601872.XSHG', '300570.XSHE', '300487.XSHE', '300725.XSHE', '300496.XSHE', '000425.XSHE', '000975.XSHE', '300383.XSHE', '300596.XSHE', '600125.XSHG', '600352.XSHG', '600745.XSHG', '600763.XSHG', '000830.XSHE', '000877.XSHE', '002023.XSHE', '300750.XSHE', '600570.XSHG', '600782.XSHG', '603558.XSHG', '000026.XSHE', '300274.XSHE', '300760.XSHE', '600622.XSHG', '000338.XSHE', '000858.XSHE', '000977.XSHE', '002111.XSHE', '002189.XSHE', '601100.XSHG', '603259.XSHG']
sell_stock []
-----------买入此次调仓

In [ ]:
import matplotlib

# 设置后端为 'Qt5Agg'

matplotlib.use('Qt5Agg')

b = cerebro.plot(iplot=False, filename='output.png')
b[0][0]

In [16]:

# 提取收益序列
pnl = pd.Series(result[0].analyzers._TimeReturn.get_analysis())
# 计算累计收益
cumulative = (pnl + 1).cumprod()
# 计算回撤序列
max_return = cumulative.cummax()
drawdown = (cumulative - max_return) / max_return

# 计算收益评价指标
import pyfolio as pf
# 按年统计收益指标
perf_stats_year = (pnl).groupby(pnl.index.to_period('y')).apply(lambda data: pf.timeseries.perf_stats(data)).unstack()
# 统计所有时间段的收益指标
perf_stats_all = pf.timeseries.perf_stats((pnl)).to_frame(name='all')
perf_stats = pd.concat([perf_stats_year, perf_stats_all.T], axis=0)
perf_stats_ = round(perf_stats,4).reset_index()


# 绘制图形
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号
import matplotlib.ticker as ticker # 导入设置坐标轴的模块
#plt.style.use('seaborn') 
#plt.style.use('dark_background')

fig, (ax0, ax1) = plt.subplots(2,1, gridspec_kw = {'height_ratios':[1.5, 4]}, figsize=(20,8))
cols_names = ['date', 'Annual\nreturn', 'Cumulative\nreturns', 'Annual\nvolatility',
             'Sharpe\nratio', 'Calmar\nratio', 'Stability', 'Max\ndrawdown',
             'Omega\nratio', 'Sortino\nratio', 'Skew', 'Kurtosis', 'Tail\nratio',
             'Daily value\nat risk']

# 绘制表格
ax0.set_axis_off() # 除去坐标轴
table = ax0.table(cellText = perf_stats_.values,
                bbox=(0,0,1,1), # 设置表格位置， (x0, y0, width, height)
                rowLoc = 'right', # 行标题居中
                cellLoc='right' ,
                colLabels = cols_names, # 设置列标题
                colLoc = 'right', # 列标题居中
                edges = 'open' # 不显示表格边框
                )
table.set_fontsize(13)

# 绘制累计收益曲线
ax2 = ax1.twinx()
ax1.yaxis.set_ticks_position('right') # 将回撤曲线的 y 轴移至右侧
ax2.yaxis.set_ticks_position('left') # 将累计收益曲线的 y 轴移至左侧
# 绘制回撤曲线
drawdown.plot.area(ax=ax1, label='drawdown (right)', rot=0, alpha=0.3, fontsize=13, grid=False)
# 绘制累计收益曲线
(cumulative).plot(ax=ax2, color='#F1C40F' , lw=3.0, label='cumret (left)', rot=0, fontsize=13, grid=False)
# 不然 x 轴留有空白
ax2.set_xbound(lower=cumulative.index.min(), upper=cumulative.index.max())
# 主轴定位器：每 5 个月显示一个日期：根据具体天数来做排版
ax2.xaxis.set_major_locator(ticker.MultipleLocator(100))
# 同时绘制双轴的图例
h1,l1 = ax1.get_legend_handles_labels()
h2,l2 = ax2.get_legend_handles_labels()
plt.legend(h1+h2,l1+l2, fontsize=12, loc='upper left', ncol=1)

fig.tight_layout() # 规整排版
plt.show()

<IPython.core.display.Javascript object>

In [55]:
import matplotlib.pyplot as plt

def find_nearest_trading_date(trading_dates, target_date, latter=False):
    # 将目标日期转换为Timestamp对象
    target_date = pd.Timestamp(target_date)

    # 遍历交易日期列表，找到最接近的日期
    nearest_date = min(trading_dates, key=lambda date: abs(date - target_date))
    if latter:
        if nearest_date < target_date:
            nearest_date = trading_dates[trading_dates.get_loc(nearest_date) + 1]
    return nearest_date, trading_dates.get_loc(nearest_date)

class context(object):
    def __init__(self, all_gotPrice, tradeDates):
        self.portfolio = {}
        self.portfolio_value = []
        self.event_log = pd.DataFrame(columns=['date', 'stock', 'event'])
        self.start_date = pd.to_datetime('2019-01-31')
        self.end_date = pd.to_datetime('2024-05-16')
        self.cash = 10000000000000000000
        self.commision = 0.003
        self.tax = 0.001
        self.tradeDates = tradeDates[tradeDates.get_loc(self.start_date):tradeDates.get_loc(self.end_date)+1]
        self.all_gotPrice = all_gotPrice

    def buy(self, stock, date, num, prices:pd.DataFrame):
        #prices = self.all_gotPrice[(self.all_gotPrice['order_book_id'] == stock) & (self.all_gotPrice['date'] == date)]
        if prices.empty:
            self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': ['无数据']})], axis=0)
            return False
        open_price = prices['open']
        high_price = prices['high']
        low_price = prices['low']
        close_price = prices['close']

        if open_price == high_price == low_price == close_price:
            self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': ['涨跌停']})], axis=0)
            return False
        if num * open_price > self.cash:
            self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': ['资金不足']})], axis=0)
            return False
        else:
            self.cash -= num * open_price * (1 + self.commision)
            if stock in self.portfolio:
                self.portfolio[stock] += num
            else:
                self.portfolio[stock] = num
            self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': [f'买入{num}']})], axis=0)
            return True

    def sell(self, stock, date, num, prices:pd.DataFrame):
        #prices = self.all_gotPrice[(self.all_gotPrice['order_book_id'] == stock) & (self.all_gotPrice['date'] == date)]
        if prices.empty:
            self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': ['无数据']})], axis=0)
            return False
        open_price = prices['open']
        high_price = prices['high']
        low_price = prices['low']
        close_price = prices['close']

        if open_price == high_price == low_price == close_price:
            self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': ['涨跌停']})], axis=0)
            return False
        if stock not in self.portfolio:
            self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': ['未持有']})], axis=0)
            return False
        else:
            self.cash += num * open_price * (1 - self.commision - self.tax)
            self.portfolio[stock] -= num
            if self.portfolio[stock] == 0:
                self.portfolio.pop(stock)
            self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': [f'卖出{num}']})], axis=0)
            return True

    def get_portfolio_value1(self, date):
        portfolio_value = self.cash
        for stock in self.portfolio:
            prices = self.all_gotPrice[(self.all_gotPrice['order_book_id'] == stock) & (self.all_gotPrice['date'] == date)]
            close_price = prices['close'].values[0]
            portfolio_value += self.portfolio[stock] * close_price
        return portfolio_value
    
    def get_portfolio_value(self, date):
        # 提取指定日期的所有价格数据，并将 `order_book_id` 设置为索引
        prices_on_date = self.all_gotPrice[self.all_gotPrice['date'] == date].set_index('order_book_id')['close']
        
        portfolio_value = self.cash
        for stock, quantity in self.portfolio.items():
            # 直接从已设置好索引的 Series 中获取价格
            try:
                close_price = prices_on_date[stock]
                portfolio_value += quantity * close_price
            except KeyError:
                print(f'{stock}在{date}停牌')
                continue
            except Exception as e:
                print(f'get_portfolio_value中{stock}在{date}出现{e}')
                continue
        return portfolio_value

    def adjust_portfolio(self, date, targetList: list):
        self.portfolio_value = self.get_portfolio_value(date)
        average_target_value = self.portfolio_value / len(targetList)
        prices = self.all_gotPrice[self.all_gotPrice['date'] == date].set_index('order_book_id')
        # 多退少补，先卖再买
        for stock in list(self.portfolio.keys()):
            try:
                target_prices = prices.loc[stock]
                if stock not in targetList:
                    self.sell(stock, date, self.portfolio[stock], target_prices)
            except Exception as e:
                print(f'{stock}在{date}出现{e}')
                continue
        # sell
        for stock in targetList:
            try:
                target_prices = prices.loc[stock]
                target_num = average_target_value / target_prices['open']
                if target_num < self.portfolio[stock]:
                    sell_num = round(self.portfolio[stock] - target_num, 0)
                    self.sell(stock, date, sell_num, target_prices)

            except Exception as e:
                print(f'{stock}在{date}出现{e}')
                continue
        
        # buy
        for stock in targetList:
            try:
                target_prices = prices.loc[stock]
                target_num = average_target_value / target_prices['open']
                if stock in self.portfolio:
                    if target_num >= self.portfolio[stock]:
                        if stock[0:3] == '688': # 科创板200股起买
                            buy_num = 200 + 100 * (target_num - self.portfolio[stock] - 200 // 100)
                            if buy_num < 200:
                                self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': ['科创板不足200']})], axis=0)
                                continue
                            self.buy(stock, date, buy_num, target_prices)
                        else:
                            buy_num = 100 * (target_num - self.portfolio[stock] // 100)
                            self.buy(stock, date, buy_num, target_prices)
                else:
                    if stock[0:3] == '688': #科创板200股起买
                        buy_num = 200 + 100 * (target_num - 200 // 100)
                        if buy_num < 200:
                            self.event_log = pd.concat([self.event_log, pd.DataFrame({'date': [date], 'stock': [stock], 'event': ['科创板不足200']})], axis=0)
                            continue
                        self.buy(stock, date, buy_num, target_prices)
                    else:
                        buy_num = 100 * (target_num // 100)
                        self.buy(stock, date, buy_num, target_prices)
            except Exception as e:
                print(f'{stock}在{date}出现{e}')
                continue

portfolio_values = []
portfolio_history = []

portfolio_df = pd.read_csv('portfolio1/portfolio_df.csv')
portfolio_df['Date'] = pd.to_datetime(portfolio_df['Date'])
adjust_dates = portfolio_df['Date'].tolist()

context = context(all_gotPrice, tradeDates)
for date in tqdm_notebook(context.tradeDates):
    if date in adjust_dates: #调仓日
        adjust_date = find_nearest_trading_date(tradeDates, date, latter=True)[0] #下一个交易日调仓
        targetList = [stock for stock in portfolio_df.loc[portfolio_df['Date'] == date].values.tolist()[0][1:] if type(stock) == str]
        context.adjust_portfolio(adjust_date, targetList)
    portfolio_values.append(context.get_portfolio_value(date))
    portfolio_history.append(context.portfolio)

plt.figure(figsize=(20, 10), dpi=150)
plt.plot(context.tradeDates, portfolio_values)

C:\Users\stansfield\AppData\Local\Temp\ipykernel_29368\556520327.py:164: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for date in tqdm_notebook(context.tradeDates):


  0%|          | 0/1280 [00:00<?, ?it/s]

600519.XSHG在2019-01-31 00:00:00出现'600519.XSHG'
300347.XSHE在2019-01-31 00:00:00出现'300347.XSHE'
300073.XSHE在2019-01-31 00:00:00出现'300073.XSHE'
600699.XSHG在2019-01-31 00:00:00出现'600699.XSHG'
002299.XSHE在2019-01-31 00:00:00出现'002299.XSHE'
002916.XSHE在2019-01-31 00:00:00出现'002916.XSHE'
000401.XSHE在2019-01-31 00:00:00出现'000401.XSHE'
002475.XSHE在2019-01-31 00:00:00出现'002475.XSHE'
300699.XSHE在2019-01-31 00:00:00出现'300699.XSHE'
300394.XSHE在2019-01-31 00:00:00出现'300394.XSHE'
300457.XSHE在2019-01-31 00:00:00出现'300457.XSHE'
600567.XSHG在2019-01-31 00:00:00出现'600567.XSHG'
600845.XSHG在2019-01-31 00:00:00出现'600845.XSHG'
300673.XSHE在2019-01-31 00:00:00出现'300673.XSHE'
300628.XSHE在2019-01-31 00:00:00出现'300628.XSHE'
000539.XSHE在2019-01-31 00:00:00出现'000539.XSHE'
000672.XSHE在2019-01-31 00:00:00出现'000672.XSHE'
002353.XSHE在2019-01-31 00:00:00出现'002353.XSHE'
300575.XSHE在2019-01-31 00:00:00出现'300575.XSHE'
300735.XSHE在2019-01-31 00:00:00出现'300735.XSHE'
601012.XSHG在2019-01-31 00:00:00出现'601012.XSHG'
601668.XSHG在2

In [54]:
plt.figure(figsize=(20, 10), dpi=150)
plt.plot(context.tradeDates, portfolio_values)

In [27]:
from math import nan


portfolio_df = pd.read_csv('portfolio1/portfolio_df.csv')
portfolio_df['Date'] = pd.to_datetime(portfolio_df['Date'])

a = [stock for stock in portfolio_df.loc[portfolio_df['Date'] == '2019-01-31'].values.tolist()[0][1:] if type(stock) == str]
a

['600519.XSHG',
 '300347.XSHE',
 '300073.XSHE',
 '600699.XSHG',
 '002299.XSHE',
 '002916.XSHE',
 '000401.XSHE',
 '002475.XSHE',
 '300699.XSHE',
 '300394.XSHE',
 '300457.XSHE',
 '600567.XSHG',
 '600845.XSHG',
 '300673.XSHE',
 '300628.XSHE',
 '000539.XSHE',
 '000672.XSHE',
 '002353.XSHE',
 '300575.XSHE',
 '300735.XSHE',
 '601012.XSHG',
 '601668.XSHG',
 '603338.XSHG',
 '603588.XSHG',
 '002798.XSHE',
 '002831.XSHE',
 '300284.XSHE',
 '600019.XSHG',
 '601058.XSHG',
 '601872.XSHG',
 '300570.XSHE',
 '300487.XSHE',
 '300725.XSHE',
 '300496.XSHE',
 '000425.XSHE',
 '000975.XSHE',
 '300383.XSHE',
 '300596.XSHE',
 '600125.XSHG',
 '600352.XSHG',
 '600745.XSHG',
 '600763.XSHG',
 '000830.XSHE',
 '000877.XSHE',
 '002023.XSHE',
 '300750.XSHE',
 '600570.XSHG',
 '600782.XSHG',
 '603558.XSHG',
 '000026.XSHE',
 '300274.XSHE',
 '300760.XSHE',
 '600622.XSHG',
 '000338.XSHE',
 '000858.XSHE',
 '000977.XSHE',
 '002111.XSHE',
 '002189.XSHE',
 '601100.XSHG',
 '603259.XSHG']